# ⚖️ Islamic Fatwa RAG — Evaluation & LLM Judge Benchmark
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbderrahimZineddineEsi/BinBaz-RAG/blob/multi-sheikh-rag/notebooks/04_evaluation_and_benchmarking.ipynb)

A complete evaluation and model benchmarking pipeline designed to:
1. **Sample 100 Multi-Scholar Fatwa Responses**: Collect top 2 candidate answers per scholar across 7 Sunni / Salafi authorities using Mode 1 retrieval (`x_per_scholar=2`) to capture true answers (`answers-it`), near-misses (`related-only`), and distractors (`unrelated`).
2. **Interactive Human Evaluation Card Deck**: Grade fatwa answers one by one with a dedicated 1-click interactive card interface that continuously auto-saves to disk.
3. **Multi-Model Benchmark**: Evaluate classification performance across:
   - **Laya Multilingual** (`convaiinnovations/laya-multilingual` local 322M mmBERT)
   - **JEV** (`typesafe/jev-1.13` via OpenRouter Decisions API)
   - **Gemini Family** (`gemini-2.5-flash`, `gemini-2.5-pro`, `gemini-3.1-flash-lite`, `gemini-3.5-flash`)
   - **Claude Family** (`anthropic/claude-3.5-sonnet`, `anthropic/claude-3.7-sonnet`)
   - **Open Models** (`deepseek/deepseek-chat`, `openai/gpt-4o-mini`, `qwen/qwen-2.5-72b-instruct`)
4. **Metrics & Economics Dashboard**: Compute Accuracy, Macro F1, Per-Class F1, Confusion Matrices, Speed (ms/query), and Cost per 100 items + projected cost for mass-labeling 10,000 items.
5. **Laya Fine-Tuning Export**: Export labeled dataset to Laya's native training JSONL format for fine-tuning your local decision engine.

In [1]:
# 1. Environment Setup & Key Verification
import os
import sys
import time
import json
import re
from pathlib import Path
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from dotenv import load_dotenv

# Auto-detect project root and data directory reliably
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "app.py").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "app.py").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
elif Path("/content/binbaz-rag").exists():
    PROJECT_ROOT = Path("/content/binbaz-rag")
else:
    PROJECT_ROOT = Path("/home/zeno/Documents/work/FULL_PROJECTS/binbaz-rag")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"

def get_data_file(filename: str) -> Path:
    """Finds existing file in data/ or notebooks/data/, falling back to data/."""
    candidates = [
        PROJECT_ROOT / "data" / filename,
        PROJECT_ROOT / "notebooks" / "data" / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    return DATA_DIR / filename

# Try importing Google Colab userdata secrets
try:
    from google.colab import userdata
    for k in ("GEMINI_API_KEY", "OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            val = userdata.get(k)
            if val:
                os.environ[k] = val
        except Exception:
            pass
except ImportError:
    pass

load_dotenv()

print("=" * 60)
print("ENVIRONMENT STATUS")
print("=" * 60)
cuda_ok = torch.cuda.is_available()
print(f"CUDA Available:     {cuda_ok}")
if cuda_ok:
    print(f"GPU Device:         {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM:         {vram_gb:.2f} GB")

print(f"Gemini API Key:     {'✓ Present' if os.getenv('GEMINI_API_KEY') else '✕ Missing (Colab Secrets / .env)'}")
print(f"OpenRouter API Key: {'✓ Present' if os.getenv('OPENROUTER_API_KEY') else '✕ Missing (Colab Secrets / .env)'}")
print(f"Project Root:       {PROJECT_ROOT}")
print(f"Data Directory:     {DATA_DIR}")
print("=" * 60)


ENVIRONMENT STATUS
CUDA Available:     True
GPU Device:         NVIDIA GeForce GTX 1050
Total VRAM:         2.94 GB
Gemini API Key:     ✓ Present
OpenRouter API Key: ✓ Present


In [ ]:
# 2. Load Multi-Scholar Index & Retrieval Pipeline
# (Skipped automatically if candidate dataset already exists on disk to save time & memory!)
from app import FatwaApp, Settings, ensure_data

CANDIDATES_JSON = get_data_file("evaluation_candidates_100.json")
FORCE_RELOAD_CORPUS = False  # Set to True if you want to force loading the corpus and embedder/reranker

app = None
if CANDIDATES_JSON.exists() and not FORCE_RELOAD_CORPUS:
    print("=" * 60)
    print(f"✓ Existing candidate dataset found at: {CANDIDATES_JSON}")
    print("✓ Skipping heavy embedder/reranker loading to save time & VRAM!")
    print("  (If you want to resample fresh candidates from scratch, set FORCE_RELOAD_CORPUS = True)")
    print("=" * 60)
else:
    ensure_data(DATA_DIR)
    # Device allocation: if GPU has <= 6GB VRAM, run embedder on CPU and reranker on CUDA to avoid OOM
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
    device_embed = "cpu" if vram_gb < 6 else ("cuda" if torch.cuda.is_available() else "cpu")
    device_rerank = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading embedder on [{device_embed}] and reranker on [{device_rerank}]...")
    settings = Settings(
        data_dir=DATA_DIR,
        device_embedder=device_embed,
        device_reranker=device_rerank,
        warmup=False,
    )
    app = FatwaApp(settings)

    print(f"\n✓ Corpus loaded: {len(app.df):,} fatwas across {len(app.available_scholars)} scholars:")
    for s, count in app.df["source"].value_counts().items():
        print(f"  • {s:25s}: {count:,} fatwas")


ModuleNotFoundError: No module named 'app'

In [ ]:
# 3. Candidate Sampling / Dataset Loader
# If you already have evaluation_candidates_100.json, it loads directly in milliseconds.
# This guarantees candidate IDs (1..100) stay 100% synchronized with your human ground-truth labels!

CANDIDATES_JSON = get_data_file("evaluation_candidates_100.json")
CANDIDATES_CSV = get_data_file("evaluation_candidates_100.csv")

# Set REGENERATE_CANDIDATES = True ONLY if you want to wipe current candidates and sample new ones
REGENERATE_CANDIDATES = False

# Diverse jurisprudential questions (worship, transactions, oaths, family law, nullifiers)
DEFAULT_QUESTIONS = [
    "ما هي كفارة اليمين ومتى يشرع الإطعام ومتى يصام؟",
    "ما حكم إخراج زكاة الفطر نقداً بدل الطعام للفقراء؟",
    "هل صلاة الجماعة في المسجد واجبة عيناً على كل رجل قادر؟",
    "ما حكم الجمع بين صلاتي الظهر والعصر أو المغرب والعشاء لعذر المطر أو الوحل؟",
    "ما هي شروط الحجاب الشرعي الكامل للمرأة أمام الرجال الأجانب؟",
    "هل ينتقض الوضوء بلمس المرأة أو أكل لحم الإبل؟",
    "ما حكم عقود التأمين التجاري على السيارات والتأمين على الحياة؟",
    "ما حكم الطلاق في حال الغضب الشديد وهل يقع؟",
]

EVAL_QUESTIONS = DEFAULT_QUESTIONS

def sample_evaluation_candidates(app_instance: FatwaApp, questions: list[str], target_count: int = 100, x_per_scholar: int = 2) -> pd.DataFrame:
    if app_instance is None:
        raise ValueError("FatwaApp is not loaded! Set FORCE_RELOAD_CORPUS = True in Cell 2 to sample new candidates.")
    records = []
    print(f"Sampling top {x_per_scholar} fatwas per scholar across {len(questions)} questions...")
    
    for q_idx, query in enumerate(questions, 1):
        cands_by_scholar = app_instance.retrieve_top_k_per_scholar(
            query, 
            app_instance.available_scholars, 
            pool_per_scholar=max(x_per_scholar * 2, 4)
        )
        
        for scholar in app_instance.available_scholars:
            df_s = cands_by_scholar.get(scholar)
            if df_s is None or len(df_s) == 0:
                continue
            
            # Rerank with Mizan-Rerank-V2
            reranked_s = app_instance.rerank_candidates(query, df_s).head(x_per_scholar)
            for _, row in reranked_s.iterrows():
                records.append({
                    "id": len(records) + 1,
                    "query_id": q_idx,
                    "query": query,
                    "scholar": str(row["source"]),
                    "fatwa_id": int(row["fatwa_id"]),
                    "title": str(row["title"]),
                    "question_text": str(row.get("question", "")),
                    "answer_text": str(row["answer"]),
                    "reranker_score": round(float(row["reranker_score"]), 4),
                    "embedding_score": round(float(row.get("embedding_score", 0.0)), 4),
                })
                if len(records) >= target_count:
                    break
            if len(records) >= target_count:
                break
        if len(records) >= target_count:
            break
            
    df_eval = pd.DataFrame(records)
    df_eval.to_json(CANDIDATES_JSON, orient="records", force_ascii=False, indent=2)
    df_eval.to_csv(CANDIDATES_CSV, index=False)
    
    print(f"\n✓ Collected exactly {len(df_eval)} candidate responses across {df_eval['scholar'].nunique()} scholars.")
    print(f"Saved dataset to:\n  - {CANDIDATES_JSON}\n  - {CANDIDATES_CSV}")
    return df_eval

if CANDIDATES_JSON.exists() and not REGENERATE_CANDIDATES:
    print("=" * 60)
    print(f"✓ Preserving existing candidate dataset: {CANDIDATES_JSON}")
    candidates_df = pd.read_json(CANDIDATES_JSON)
    print(f"✓ Loaded {len(candidates_df)} candidates. Skipping query re-run to preserve IDs matching your labels!")
    print("=" * 60)
else:
    candidates_df = sample_evaluation_candidates(app, EVAL_QUESTIONS, target_count=100, x_per_scholar=2)

print("\nDistribution of sampled candidates by scholar:")
print(candidates_df["scholar"].value_counts().to_string())

print("\nSample candidate cards:")
display(candidates_df[["id", "query_id", "scholar", "title", "reranker_score"]].head(10))


In [ ]:
# 4. Interactive Human Evaluation Card Deck
# Review candidate answers and classify with:
#  [answers-it]   -> Direct answer to question (Green)
#  [related-only]  -> Related topic or conditional/indirect (Yellow)
#  [unrelated]     -> Irrelevant or negative distraction (Red)
# Every decision is saved instantly to data/evaluation_ground_truth.json.

import gradio as gr

LABELS_FILE = get_data_file("evaluation_ground_truth.json")

def load_saved_labels() -> dict[str, str]:
    if LABELS_FILE.exists():
        try:
            with open(LABELS_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {}

def save_label(item_id: int, label: str):
    labels = load_saved_labels()
    labels[str(item_id)] = label
    with open(LABELS_FILE, "w", encoding="utf-8") as f:
        json.dump(labels, f, ensure_ascii=False, indent=2)

def build_card_html(item: dict, label: str | None, current_idx: int, total: int, counts: dict) -> str:
    lbl_badge = ""
    if label == "answers-it":
        lbl_badge = "<span style='background:#17794f;color:#fff;padding:4px 14px;border-radius:999px;font-weight:700;'>✓ Answers it (إجابة مباشرة)</span>"
    elif label == "related-only":
        lbl_badge = "<span style='background:#b3832a;color:#fff;padding:4px 14px;border-radius:999px;font-weight:700;'>≈ Related only (مسألة ذات صلة)</span>"
    elif label == "unrelated":
        lbl_badge = "<span style='background:#b03a3a;color:#fff;padding:4px 14px;border-radius:999px;font-weight:700;'>✕ Unrelated (غير متعلق)</span>"
    else:
        lbl_badge = "<span style='background:#5f6a66;color:#fff;padding:4px 14px;border-radius:999px;font-weight:700;'>? Unlabeled (لم يُقيَّم بعد)</span>"

    ans_excerpt = (item.get("answer_text") or "")[:1400]
    if len(item.get("answer_text") or "") > 1400:
        ans_excerpt += "…"

    return f"""
    <div style='background:#151f1c;border:1.5px solid #26352f;border-radius:18px;padding:22px;color:#e7eeeb;font-family:system-ui,sans-serif;margin:8px 0;'>
      <div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:14px;border-bottom:1px solid #26352f;padding-bottom:12px;'>
        <div>
          <span style='font-size:1.15rem;font-weight:800;color:#3dc3a0;'>Card #{current_idx + 1} of {total}</span>
          <span style='margin-left:12px;font-size:0.9rem;color:#8fa09a;'>Query ID: {item.get("query_id")}</span>
        </div>
        <div>{lbl_badge}</div>
      </div>
      
      <div style='margin-bottom:14px;'>
        <div style='font-size:0.75rem;font-weight:800;text-transform:uppercase;color:#8fa09a;letter-spacing:0.08em;margin-bottom:4px;'>السؤال (Question)</div>
        <div style='font-size:1.35rem;font-weight:700;color:#fff;direction:rtl;text-align:right;font-family:serif;'>{item.get("query")}</div>
      </div>

      <div style='margin-bottom:14px;display:flex;gap:10px;align-items:center;flex-wrap:wrap;'>
        <span style='background:#143229;color:#6fdbbf;border:1px solid #3dc3a0;padding:3px 12px;border-radius:999px;font-weight:800;font-size:0.9rem;'>
          الشيخ / الهيئة: {item.get("scholar")}
        </span>
        <span style='background:#222e2a;color:#a2b0aa;padding:3px 10px;border-radius:999px;font-size:0.85rem;'>
          Mizan Score: {item.get("reranker_score")}
        </span>
        <span style='background:#222e2a;color:#a2b0aa;padding:3px 10px;border-radius:999px;font-size:0.85rem;'>
          Fatwa #{item.get("fatwa_id")}
        </span>
      </div>

      <div style='margin-bottom:12px;'>
        <div style='font-size:0.75rem;font-weight:800;text-transform:uppercase;color:#8fa09a;letter-spacing:0.08em;margin-bottom:4px;'>عنوان الفتوى (Title)</div>
        <div style='font-size:1.18rem;font-weight:700;color:#e2b658;direction:rtl;text-align:right;font-family:serif;'>{item.get("title")}</div>
      </div>

      <div>
        <div style='font-size:0.75rem;font-weight:800;text-transform:uppercase;color:#8fa09a;letter-spacing:0.08em;margin-bottom:4px;'>نص الإجابة (Answer Excerpt)</div>
        <div style='background:#1b2824;border:1px solid #26352f;border-radius:12px;padding:14px;font-size:1.12rem;line-height:1.75;direction:rtl;text-align:right;max-height:260px;overflow-y:auto;color:#e7eeeb;font-family:serif;'>
          {ans_excerpt}
        </div>
      </div>

      <div style='margin-top:14px;padding-top:10px;border-top:1px solid #26352f;display:flex;justify-content:space-between;font-size:0.85rem;color:#8fa09a;'>
        <span>Labeled: <b>{counts.get("total_labeled", 0)}/{total}</b> ({counts.get("pct", 0)}%)</span>
        <span>Answers: <b style='color:#58d69a;'>{counts.get("answers-it", 0)}</b> | Related: <b style='color:#e8bb55;'>{counts.get("related-only", 0)}</b> | Unrelated: <b style='color:#f08a80;'>{counts.get("unrelated", 0)}</b></span>
      </div>
    </div>
    """

# Check if ground truth already exists or generate initial baseline labels
def get_label_counts(items, saved_labels):
    c = {"answers-it": 0, "related-only": 0, "unrelated": 0}
    for item in items:
        lbl = saved_labels.get(str(item["id"]))
        if lbl in c:
            c[lbl] += 1
    total_lbl = sum(c.values())
    c["total_labeled"] = total_lbl
    c["pct"] = round(100 * total_lbl / max(1, len(items)), 1)
    return c

items_list = candidates_df.to_dict(orient="records")
saved_labels = load_saved_labels()

# Helper: generate starter labels using heuristic thresholds if starting fresh
def generate_heuristic_ground_truth(candidates: pd.DataFrame, force: bool = False):
    labels = load_saved_labels()
    if labels and not force:
        print(f"Found {len(labels)} existing human labels in {LABELS_FILE}.")
        return labels
    
    print("Generating starter heuristic ground truth (high Mizan score -> answers-it, medium -> related-only, low -> unrelated)...")
    starter = {}
    for _, r in candidates.iterrows():
        score = float(r["reranker_score"])
        if score >= 0.85:
            lbl = "answers-it"
        elif score >= 0.40:
            lbl = "related-only"
        else:
            lbl = "unrelated"
        starter[str(r["id"])] = lbl
        
    with open(LABELS_FILE, "w", encoding="utf-8") as f:
        json.dump(starter, f, ensure_ascii=False, indent=2)
    print(f"✓ Generated {len(starter)} baseline labels in {LABELS_FILE}. You can review & modify each card in the UI below!")
    return starter

# Uncomment the line below if you want to immediately pre-fill starter labels for testing:
# generate_heuristic_ground_truth(candidates_df, force=False)

print("Starting Human Annotation Interface...")
with gr.Blocks(title="Fatwa Evaluation Cards") as eval_deck:
    cur_idx = gr.State(0)
    
    card_view = gr.HTML()
    
    with gr.Row():
        btn_answers = gr.Button("✓ Answers it (إجابة مباشرة)", variant="primary", scale=1)
        btn_related = gr.Button("≈ Related only (مسألة ذات صلة)", scale=1)
        btn_unrelated = gr.Button("✕ Unrelated (غير متعلق)", variant="stop", scale=1)
        
    with gr.Row():
        btn_prev = gr.Button("◀ Previous Card", size="sm", scale=1)
        slider_jump = gr.Slider(1, len(items_list), value=1, step=1, label="Jump to Card #", scale=2)
        btn_next = gr.Button("Next Card ▶", size="sm", scale=1)

    def render_card(idx: int):
        idx = max(0, min(idx, len(items_list) - 1))
        item = items_list[idx]
        labels = load_saved_labels()
        lbl = labels.get(str(item["id"]))
        counts = get_label_counts(items_list, labels)
        html = build_card_html(item, lbl, idx, len(items_list), counts)
        return html, idx + 1, idx

    def handle_label(label: str, idx: int):
        if 0 <= idx < len(items_list):
            item = items_list[idx]
            save_label(item["id"], label)
        next_idx = min(idx + 1, len(items_list) - 1)
        return render_card(next_idx)

    btn_answers.click(lambda idx: handle_label("answers-it", idx), inputs=[cur_idx], outputs=[card_view, slider_jump, cur_idx])
    btn_related.click(lambda idx: handle_label("related-only", idx), inputs=[cur_idx], outputs=[card_view, slider_jump, cur_idx])
    btn_unrelated.click(lambda idx: handle_label("unrelated", idx), inputs=[cur_idx], outputs=[card_view, slider_jump, cur_idx])

    btn_prev.click(lambda idx: render_card(idx - 1), inputs=[cur_idx], outputs=[card_view, slider_jump, cur_idx])
    btn_next.click(lambda idx: render_card(idx + 1), inputs=[cur_idx], outputs=[card_view, slider_jump, cur_idx])
    slider_jump.change(lambda val: render_card(int(val) - 1), inputs=[slider_jump], outputs=[card_view, slider_jump, cur_idx])

    eval_deck.load(lambda: render_card(0), outputs=[card_view, slider_jump, cur_idx])

# Launch inline inside notebook
eval_deck.launch(inline=True, height=720, quiet=True)


In [ ]:
# 5. Model Predictor Implementations
# Unified interface returning: verdict (answers-it, related-only, unrelated), confidence, latency_s, tokens, cost

import requests

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
DECISIONS_URL = "https://openrouter.ai/api/alpha/decisions"

# --- 1. Laya-Multilingual (Local Decision Engine) ---
_LAYA_AGENT = None

def get_laya_agent():
    global _LAYA_AGENT
    if _LAYA_AGENT is None:
        import laya
        print("Loading local laya-multilingual model...")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _LAYA_AGENT = laya.load("convaiinnovations/laya-multilingual", device=device)
        print("✓ Laya loaded.")
    return _LAYA_AGENT

def predict_laya(item: dict) -> dict:
    agent = get_laya_agent()
    state = {
        "user_query": item["query"],
        "scholar": item["scholar"],
        "fatwa_title": item["title"],
        "fatwa_answer": item["answer_text"][:1200],
    }
    questions = {
        "relevance": {
            "type": "choice",
            "instructions": (
                "Judge whether the fatwa in `fatwa_answer` directly answers `user_query`. "
                "Select 'answers-it' if it provides the requested ruling/answer. "
                "Select 'related-only' if it discusses a related subject without answering the query. "
                "Select 'unrelated' if it does not address the query."
            ),
            "criteria": {
                "answers-it": "The fatwa directly answers the question or gives the ruling requested.",
                "related-only": "The fatwa is on a related Islamic topic but does not answer the specific question.",
                "unrelated": "The fatwa does not address the question.",
            }
        }
    }
    t0 = time.perf_counter()
    res = agent.predict(state, questions)
    dt = time.perf_counter() - t0
    
    ans = res.get("answers", {}).get("relevance", {})
    choice = ans.get("choice", "unknown")
    conf = ans.get("confidence", 0.0)
    usage = res.get("usage", {})
    
    return {
        "verdict": choice,
        "confidence": float(conf) if conf is not None else 1.0,
        "latency_s": dt,
        "prompt_tokens": usage.get("input_tokens", 0),
        "completion_tokens": usage.get("output_tokens", 0),
        "cost": 0.0,  # Local execution has $0 API cost!
    }

# --- 2. JEV (OpenRouter Decisions API) ---
def predict_jev(item: dict, model_id: str = "typesafe/jev-1.13") -> dict:
    key = os.getenv("OPENROUTER_API_KEY")
    if not key:
        raise ValueError("OPENROUTER_API_KEY required for JEV.")
        
    state = {
        "user_query": item["query"],
        "scholar": item["scholar"],
        "fatwa_title": item["title"],
        "fatwa_answer": item["answer_text"][:1200],
    }
    questions = {
        "relevance": {
            "type": "choice",
            "instructions": "Judge whether `fatwa_answer` answers `user_query`.",
            "criteria": {
                "answers-it": "Directly answers the user's specific question.",
                "related-only": "Touches on a related Islamic topic without directly answering.",
                "unrelated": "Does not address the user's question.",
            }
        }
    }
    payload = {"model": model_id, "state": state, "questions": questions}
    headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    
    t0 = time.perf_counter()
    resp = requests.post(DECISIONS_URL, headers=headers, json=payload, timeout=60)
    dt = time.perf_counter() - t0
    
    if resp.status_code >= 400:
        raise RuntimeError(f"JEV Error {resp.status_code}: {resp.text[:300]}")
        
    data = resp.json()
    ans = data.get("answers", {}).get("relevance", {})
    u = data.get("usage", {}) or {}
    
    return {
        "verdict": ans.get("choice", "unknown"),
        "confidence": float(ans.get("confidence", 1.0)),
        "latency_s": dt,
        "prompt_tokens": u.get("input_tokens", 0),
        "completion_tokens": u.get("output_tokens", 0),
        "cost": float(u.get("cost", 0.0003)),
    }

# --- 3. LLMs (OpenRouter Chat Completions: Gemini, Claude, GPT, DeepSeek) ---
def predict_openrouter_llm(item: dict, model_id: str, temperature: float = 0.0) -> dict:
    key = os.getenv("OPENROUTER_API_KEY")
    if not key:
        raise ValueError("OPENROUTER_API_KEY required.")
        
    sys_prompt = (
        "You are an expert Islamic fatwa relevance evaluation judge. "
        "Your task is to classify whether the provided Islamic fatwa text directly answers the user's question.\n"
        "Return a JSON object with:\n"
        "- 'verdict': exactly one of ['answers-it', 'related-only', 'unrelated']\n"
        "- 'confidence': float between 0.0 and 1.0\n"
        "- 'explanation': brief Arabic explanation."
    )
    user_prompt = (
        f"User Question: {item['query']}\n\n"
        f"Scholar: {item['scholar']}\n"
        f"Title: {item['title']}\n"
        f"Fatwa Text: {item['answer_text'][:1400]}"
    )
    
    payload = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "response_format": {"type": "json_object"},
    }
    headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    
    t0 = time.perf_counter()
    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=90)
    
    # Retry without response_format if model does not support json_object mode
    if resp.status_code >= 400 and "response_format" in payload:
        del payload["response_format"]
        resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=90)
        
    dt = time.perf_counter() - t0
    
    if resp.status_code >= 400:
        raise RuntimeError(f"OpenRouter Error ({model_id}) {resp.status_code}: {resp.text[:300]}")
        
    data = resp.json()
    content = data["choices"][0]["message"]["content"]
    u = data.get("usage", {}) or {}
    
    # Parse verdict from content
    verdict = "unknown"
    conf = 1.0
    try:
        # Extract json object
        m = re.search(r"\{.*?\}", content, re.DOTALL)
        if m:
            parsed = json.loads(m.group(0))
            v = str(parsed.get("verdict", "")).lower().replace("_", "-")
            if v in ("answers-it", "related-only", "unrelated"):
                verdict = v
            conf = float(parsed.get("confidence", 1.0))
    except Exception:
        pass
        
    if verdict == "unknown":
        c_low = content.lower()
        if "answers-it" in c_low or "answers it" in c_low:
            verdict = "answers-it"
        elif "related-only" in c_low or "related only" in c_low:
            verdict = "related-only"
        elif "unrelated" in c_low:
            verdict = "unrelated"

    return {
        "verdict": verdict,
        "confidence": conf,
        "latency_s": dt,
        "prompt_tokens": u.get("prompt_tokens", 0),
        "completion_tokens": u.get("completion_tokens", 0),
        "cost": float(u.get("cost", 0.0)),
    }

print("✓ Predictor functions ready: Laya, JEV, and OpenRouter LLMs.")


In [ ]:
# 6. Execute Benchmarks Across Selected Models
# All predictions are cached in data/benchmark_predictions.json to avoid re-billing or re-computing!

BENCHMARK_FILE = get_data_file("benchmark_predictions.json")

# Models to evaluate
MODELS_CONFIG = [
    {
        "id": "laya_multilingual",
        "name": "Laya-Multilingual (Local 322M)",
        "type": "laya",
        "enabled": True,
    },
    {
        "id": "jev_1_13",
        "name": "JEV Decision Engine",
        "type": "jev",
        "model_id": "typesafe/jev-1.13",
        "enabled": True,
    },
    {
        "id": "gemini_2_5_flash",
        "name": "Gemini 2.5 Flash",
        "type": "llm",
        "model_id": "google/gemini-2.5-flash",
        "enabled": True,
    },
    {
        "id": "gemini_2_5_pro",
        "name": "Gemini 2.5 Pro",
        "type": "llm",
        "model_id": "google/gemini-2.5-pro",
        "enabled": True,
    },
    {
        "id": "claude_3_5_sonnet",
        "name": "Claude 3.5 Sonnet",
        "type": "llm",
        "model_id": "anthropic/claude-3.5-sonnet",
        "enabled": True,
    },
    {
        "id": "claude_3_7_sonnet",
        "name": "Claude 3.7 Sonnet",
        "type": "llm",
        "model_id": "anthropic/claude-3.7-sonnet",
        "enabled": True,
    },
    {
        "id": "gpt_4o_mini",
        "name": "GPT-4o Mini",
        "type": "llm",
        "model_id": "openai/gpt-4o-mini",
        "enabled": True,
    },
    {
        "id": "deepseek_chat",
        "name": "DeepSeek Chat (V3)",
        "type": "llm",
        "model_id": "deepseek/deepseek-chat",
        "enabled": True,
    },
]

def load_cached_predictions() -> dict:
    if BENCHMARK_FILE.exists():
        try:
            with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {}

def save_cached_predictions(preds: dict):
    with open(BENCHMARK_FILE, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)

def run_benchmark(candidates: list[dict], models: list[dict], max_items: int = 100) -> dict:
    cached = load_cached_predictions()
    items_to_eval = candidates[:max_items]
    
    print(f"Starting benchmark across {len(items_to_eval)} candidate fatwas...")
    
    for m in models:
        if not m.get("enabled", True):
            continue
            
        m_id = m["id"]
        m_name = m["name"]
        print(f"\n--- Benchmarking: {m_name} ---")
        
        if m_id not in cached:
            cached[m_id] = {}
            
        pbar = tqdm(items_to_eval, desc=m_name)
        for item in pbar:
            k = str(item["id"])
            if k in cached[m_id] and cached[m_id][k].get("verdict") not in (None, "unknown"):
                continue  # already computed
                
            try:
                if m["type"] == "laya":
                    res = predict_laya(item)
                elif m["type"] == "jev":
                    res = predict_jev(item, m["model_id"])
                else:
                    res = predict_openrouter_llm(item, m["model_id"])
                    
                cached[m_id][k] = res
                save_cached_predictions(cached)
            except Exception as e:
                print(f"\n! Warning: Error on item #{item['id']} ({m_name}): {e}")
                cached[m_id][k] = {"verdict": "unknown", "latency_s": 0.0, "cost": 0.0, "error": str(e)}
                save_cached_predictions(cached)
                time.sleep(1)

    print("\n✓ All model benchmark predictions completed and saved to:", BENCHMARK_FILE)
    return cached

benchmark_results = run_benchmark(items_list, MODELS_CONFIG, max_items=100)


In [ ]:
# 7. Metrics, Cost & Leaderboard Dashboard
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

ground_truth = load_saved_labels()
assert len(ground_truth) > 0, "No ground truth labels found! Please label cards in Step 4 or run generate_heuristic_ground_truth()."

print(f"Loaded {len(ground_truth)} ground truth labels.")

# Filter candidates that have human ground truth
labeled_ids = [str(item["id"]) for item in items_list if str(item["id"]) in ground_truth]
y_true = [ground_truth[i] for i in labeled_ids]

classes = ["answers-it", "related-only", "unrelated"]
metrics_summary = []

confusion_matrices = {}

for m in MODELS_CONFIG:
    if not m.get("enabled", True):
        continue
        
    m_id = m["id"]
    m_name = m["name"]
    preds_dict = benchmark_results.get(m_id, {})
    
    y_pred = []
    latencies = []
    costs = []
    
    for i in labeled_ids:
        res = preds_dict.get(i, {})
        v = res.get("verdict", "unknown")
        y_pred.append(v if v in classes else "unrelated")
        latencies.append(float(res.get("latency_s", 0.0)))
        costs.append(float(res.get("cost", 0.0)))

    # Compute metrics
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, labels=classes, average="macro", zero_division=0)
    p_per, r_per, f1_per, _ = precision_recall_fscore_support(y_true, y_pred, labels=classes, average=None, zero_division=0)
    
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    confusion_matrices[m_name] = cm
    
    total_cost_100 = sum(costs)
    projected_cost_10k = (total_cost_100 / max(1, len(labeled_ids))) * 10000
    avg_latency_ms = (np.mean(latencies) if latencies else 0) * 1000
    
    metrics_summary.append({
        "Model": m_name,
        "Accuracy (%)": round(acc * 100, 2),
        "Macro F1": round(f1_macro, 3),
        "Answers-It F1": round(f1_per[0], 3),
        "Related-Only F1": round(f1_per[1], 3),
        "Unrelated F1": round(f1_per[2], 3),
        "Avg Latency (ms)": round(avg_latency_ms, 1),
        "Cost / 100 Items": f"${total_cost_100:.4f}",
        "Projected Cost / 10K": f"${projected_cost_10k:.2f}",
        "_raw_cost": total_cost_100,
        "_raw_acc": acc,
        "_raw_f1": f1_macro,
    })

df_leaderboard = pd.DataFrame(metrics_summary).sort_values("Macro F1", ascending=False).reset_index(drop=True)

print("=" * 90)
print("BENCHMARK LEADERBOARD (Ranked by Macro F1)")
print("=" * 90)
display(df_leaderboard.drop(columns=["_raw_cost", "_raw_acc", "_raw_f1"]))

# Plot Confusion Matrices
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for idx, (m_name, cm) in enumerate(confusion_matrices.items()):
    if idx >= len(axes):
        break
    ax = axes[idx]
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(m_name, fontsize=11, fontweight="bold")
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(["Answers", "Related", "Unrel."], fontsize=9)
    ax.set_yticklabels(["Answers", "Related", "Unrel."], fontsize=9)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True Ground Truth")
    
    # Annotate numbers
    for i in range(len(classes)):
        for j in range(len(classes)):
            color = "white" if cm[i, j] > cm.max() / 2 else "black"
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontweight="bold")

plt.tight_layout()
plt.show()

# Plot Accuracy vs Cost Pareto Frontier
plt.figure(figsize=(10, 5))
for _, row in df_leaderboard.iterrows():
    plt.scatter(row["_raw_cost"], row["_raw_acc"] * 100, s=120)
    plt.annotate(row["Model"], (row["_raw_cost"] + 0.0001, row["_raw_acc"] * 100 + 0.5), fontsize=9)

plt.xlabel("Cost for 100 Items ($)")
plt.ylabel("Accuracy (%)")
plt.title("Model Quality vs Cost Trade-Off (Pareto Frontier for Mass Labeling)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


In [ ]:
# 8. Scholar Breakdown: Accuracy Across Authority Sources
# How well does each model generalize across different scholars' phrasing and styles?

scholar_data = []

for item in items_list:
    item_id = str(item["id"])
    if item_id not in ground_truth:
        continue
    truth = ground_truth[item_id]
    scholar = item["scholar"]
    
    row = {"scholar": scholar, "ground_truth": truth}
    for m in MODELS_CONFIG:
        if m.get("enabled", True):
            m_id = m["id"]
            m_pred = benchmark_results.get(m_id, {}).get(item_id, {}).get("verdict", "unknown")
            row[m["name"]] = 1 if m_pred == truth else 0
            
    scholar_data.append(row)

df_scholar_perf = pd.DataFrame(scholar_data)

print("=" * 80)
print("ACCURACY BY SCHOLAR (%)")
print("=" * 80)
model_names = [m["name"] for m in MODELS_CONFIG if m.get("enabled", True)]
scholar_group = df_scholar_perf.groupby("scholar")[model_names].mean() * 100
display(scholar_group.round(1))


In [ ]:
# 9. Export for Laya Fine-Tuning & Mass Labeling Strategy
# Export human ground-truth labels directly into Laya's native training JSONL format.

LAYA_DATASET_FILE = DATA_DIR / "laya_finetune_dataset.jsonl"

laya_training_records = []
for item in items_list:
    item_id = str(item["id"])
    if item_id not in ground_truth:
        continue
        
    choice = ground_truth[item_id]
    record = {
        "state": {
            "user_query": item["query"],
            "scholar": item["scholar"],
            "fatwa_title": item["title"],
            "fatwa_answer": item["answer_text"][:1200],
        },
        "questions": {
            "relevance": {
                "type": "choice",
                "choice": choice,
            }
        }
    }
    laya_training_records.append(record)

with open(LAYA_DATASET_FILE, "w", encoding="utf-8") as f:
    for rec in laya_training_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"✓ Exported {len(laya_training_records)} training records to {LAYA_DATASET_FILE}")
print("\nSample Laya Training Record (JSON):")
print(json.dumps(laya_training_records[0], ensure_ascii=False, indent=2))

print("""
================================================================================
MASS-LABELING & FINE-TUNING ROADMAP:
1. Select the winning model from the Benchmark Leaderboard (e.g. Gemini 2.5 Flash / Claude 3.5).
2. Run that model over 5,000 - 20,000 unlabeled fatwa candidates to generate pseudo-labels with confidence >= 0.90.
3. Combine human ground-truth with high-confidence pseudo-labels in laya_finetune_dataset.jsonl.
4. Fine-tune Laya Multilingual locally:
   agent.train(dataset_path='data/laya_finetune_dataset.jsonl', epochs=3, lr=2e-5)
5. Result: A specialized, ultra-fast (50ms), 0-cost local Arabic decision engine tailored to your fatwa corpus!
================================================================================
""")
